# Build a `MarketContext` from raw quotes

**Purpose:** Connect market-data foundations to the calibration engine by turning raw quote envelopes into a reusable `MarketContext`.

**Prerequisites:** `01_foundations/market_data_and_curves.ipynb` and `02_pricing/pricing_fundamentals.ipynb`.

**What you'll learn:**

- Load reference calibration envelopes from JSON files.
- Run `calibrate(envelope_json)` and inspect the returned `CalibrationResult`.
- Persist calibrated markets and use diagnostics before solving.

This notebook walks through the analyst-facing flow for building a `MarketContext` from a JSON `CalibrationEnvelope`. It is the canonical entry point: `calibrate(envelope_json).market`.


In [1]:
import json

import sys
sys.path.insert(0, "..")

from _shared import REPOSITORY_ROOT
from finstack_quant.valuations import calibrate

# Reference calibration envelopes ship with the repository. `_shared.REPOSITORY_ROOT`
# resolves the repo root once for every example notebook.
BOOTSTRAP_DIR = REPOSITORY_ROOT / "finstack-quant/valuations/examples/market_bootstrap"

envelope_path = BOOTSTRAP_DIR / "01_usd_discount.json"
envelope_json = envelope_path.read_text()
envelope = json.loads(envelope_json)
print(f"schema: {envelope['schema']}")
print(f"plan id: {envelope['plan']['id']}")
print(f"steps: {[step['id'] for step in envelope['plan']['steps']]}")


schema: finstack_quant.calibration
plan id: usd_ois_discount
steps: ['USD-OIS']


In [2]:
result = calibrate(envelope_json)
print(f"success: {result.success}")
print(f"rmse: {result.rmse:.3e}")
print(f"max residual: {result.max_residual:.3e}")
print(f"iterations: {result.iterations}")
result.report_to_dataframe()

success: True
rmse: 4.388e-13
max residual: 7.966e-13
iterations: 474


,step_id,success,iterations,max_residual,rmse,convergence_reason
0,USD-OIS,True,474,7.965718e-13,4.387760e-13,generic bootstrap calibration converged: max_r...


In [3]:
# `result.market` is the calibrated MarketContext, ready for downstream use.
market = result.market

# Look up the discount curve we just built.
curve = market.get_discount("USD-OIS")
print(f"discount factor at t=0:  {curve.df(0.0):.6f}")
print(f"discount factor at t=1y: {curve.df(1.0):.6f}")
print(f"discount factor at t=5y: {curve.df(5.0):.6f}")

discount factor at t=0:  1.000000
discount factor at t=1y: 0.950810
discount factor at t=5y: 0.800671


## Persisting the materialized market

`result.market_json` returns the same context as a JSON snapshot. This is
useful for caching a calibrated market between processes or for diff'ing
two calibrated states.

`MarketContext` round-trips through this snapshot losslessly:
deserialize via `MarketContext.from_json(...)` (Python) or
`MarketContext::try_from(state)` (Rust).

In [4]:
# Round-trip via the materialized JSON snapshot.
snapshot_json = result.market_json
print(f"snapshot length: {len(snapshot_json)} bytes")
# Phase 2 of the notebook tour will demonstrate composing markets and
# pulling FX cross rates / bond prices / equity spots from market_data
# (the v3 envelope's flat input list).

snapshot length: 4143 bytes


## Compose markets — chained envelope

An analyst's morning bootstrap typically chains discount → hazard → base correlation
in a single envelope. `12_full_credit_desk_market.json` shows the full pattern: rates
and CDS quotes feed two calibration steps, and a tranche quote_set drives the third.
FX cross rates ride along in `market_data` as `fx_spot` entries since they're
snapshot-only data.

In [5]:
envelope_path = BOOTSTRAP_DIR / "12_full_credit_desk_market.json"
envelope_json = envelope_path.read_text()
result = calibrate(envelope_json)
print(f"success: {result.success}")
print(f"steps: {result.step_ids}")
print(f"rmse: {result.rmse:.3e}")
result.report_to_dataframe()

success: True
steps: ['CDX-NA-IG-46', 'CDX-NA-IG-46-BASE-CORR-STEP', 'USD-OIS']
rmse: 2.519e-06


,step_id,success,iterations,max_residual,rmse,convergence_reason
0,CDX-NA-IG-46,True,441,7.186167e-13,3.779215e-13,generic bootstrap calibration converged: max_r...
1,CDX-NA-IG-46-BASE-CORR-STEP,True,99,9.012116e-06,4.506390e-06,generic bootstrap calibration converged: max_r...
2,USD-OIS,True,474,7.965718e-13,4.387760e-13,generic bootstrap calibration converged: max_r...


In [6]:
market = result.market

# Every calibrated curve is retrieved from the context by id and type — including
# base correlation, which has its own typed accessor. No snapshot parsing needed.
discount = market.get_discount("USD-OIS")
hazard = market.get_hazard("CDX-NA-IG-46")
bc = market.get_base_correlation("CDX-NA-IG-46_CORR")

print(f"USD-OIS DF(5y):    {discount.df(5.0):.6f}")
print(f"CDX-IG-46 SP(5y):  {hazard.sp(5.0):.6f}")
print(f"BC at 7%:          {bc.correlation(7.0):.4f}")

USD-OIS DF(5y):    0.800671
CDX-IG-46 SP(5y):  0.999733
BC at 7%:          0.1665


## Snapshot-only data — FX, bonds, equities

FX matrices, bond prices, equity spots, and dividend schedules are not bootstrapped
today — they ride in `market_data` as `fx_spot`, `price`, and `dividend_schedule`
entries. The reference envelopes 09 / 10 / 11 demonstrate each pattern.

In [7]:
for name in ["09_fx_matrix.json", "10_bond_prices.json", "11_equity_spots_dividends.json"]:
    result = calibrate((BOOTSTRAP_DIR / name).read_text())
    print(f"{name}: success={result.success}, steps={result.step_ids}")

09_fx_matrix.json: success=True, steps=[]
10_bond_prices.json: success=True, steps=[]
11_equity_spots_dividends.json: success=True, steps=[]


In [8]:
# Bond prices are snapshot-only and live in the market JSON under the 'prices' key.
bond_market = calibrate((BOOTSTRAP_DIR / "10_bond_prices.json").read_text()).market
prices = json.loads(bond_market.to_json()).get("prices", {})
for bond_id in ["US-TREASURY-10Y-2026-05-08", "IBM-7YR-2033"]:
    p = prices[bond_id]["price"]
    print(f"{bond_id}: {p['amount']} {p['currency']}")

US-TREASURY-10Y-2026-05-08: 99.875 USD
IBM-7YR-2033: 101.250 USD


## Validate before solving

Phase 4 will add `dry_run(envelope_json)` for fast pre-flight validation
(missing dependencies, undefined quote sets, quote class mismatches) — running
structural checks in microseconds before the (much slower) solver. For now,
`validate_calibration_json` returns the canonical pretty-printed JSON and
surfaces serde-level errors; it is the only validation tool until Phase 4 lands.

In [9]:
from finstack_quant.valuations import validate_calibration_json

canonical = validate_calibration_json((BOOTSTRAP_DIR / "01_usd_discount.json").read_text())
print(canonical[:200] + "...")

{
  "$schema": "../../schemas/calibration/3/calibration.schema.json",
  "schema": "finstack_quant.calibration",
  "plan": {
    "id": "usd_ois_discount",
    "description": "USD-OIS discount curve boo...


## When envelopes fail

Phase 4 surfaces structured diagnostics for envelope failures. Two cheap pre-flight tools are `dry_run` (runs all structural checks without invoking the solver — microseconds) and `dependency_graph_json` (dumps the step DAG). Both raise `CalibrationEnvelopeError` on malformed JSON; structural failures surface as entries in `dry_run`'s report.

The cell below deliberately misspells a `quote_set` reference. `dry_run` catches it and points at the step + suggests a fix. `calibrate` would produce the same error message at runtime — but `dry_run` is faster and lets the analyst fix the envelope before the slow solve.

In [10]:
from finstack_quant.valuations import dry_run, dependency_graph_json, CalibrationEnvelopeError

broken = json.loads((BOOTSTRAP_DIR / "01_usd_discount.json").read_text())

# Find the step's actual quote_set name and deliberately corrupt it.
original_qs = broken["plan"]["steps"][0]["quote_set"]
broken["plan"]["steps"][0]["quote_set"] = original_qs + "s"  # extra 's'

report = json.loads(dry_run(json.dumps(broken)))
for err in report["errors"]:
    if err["kind"] == "undefined_quote_set":
        print(f"[{err['kind']}] step '{err['step_id']}': quote_set '{err['ref_name']}' not found. "
              f"Suggestion: '{err.get('suggestion')}'")
    else:
        print(f"[{err['kind']}] {err}")

[undefined_quote_set] step 'USD-OIS': quote_set 'usd_quotess' not found. Suggestion: 'usd_quotes'


### Catching the typed exception

If a real failure happens downstream (e.g., solver non-convergence), `CalibrationEnvelopeError` carries `kind`, `step_id`, and `details` for programmatic handling. It inherits from `RuntimeError`, so existing `except RuntimeError` catchers continue to work.

In [11]:
# Demonstrate the typed exception by parsing intentionally malformed JSON.
try:
    dry_run("{ malformed")
except CalibrationEnvelopeError as exc:
    print(f"caught {exc.__class__.__name__}: kind={exc.kind}, step={exc.step_id}")
    payload = json.loads(exc.details)
    print(f"details kind: {payload['kind']}")

caught CalibrationEnvelopeError: kind=json_parse, step=None
details kind: json_parse


### Inspecting the dependency graph

`dependency_graph_json` returns the static DAG of a plan's steps — useful for visualizing layered envelopes (envelope 12) or debugging missing dependencies.

In [12]:
graph = json.loads(
    dependency_graph_json((BOOTSTRAP_DIR / "12_full_credit_desk_market.json").read_text())
)
print(f"initial_ids ({len(graph['initial_ids'])}): {graph['initial_ids']}")
for node in graph["nodes"]:
    reads = ", ".join(node["reads"]) or "(none)"
    writes = ", ".join(node["writes"]) or "(none)"
    print(f"step[{node['step_index']}] {node['step_id']} ({node['kind']}): "
          f"reads [{reads}] -> writes [{writes}]")

initial_ids (2): ['CDX-NA-IG-46', 'CDX-NA-IG-46_CORR']
step[0] USD-OIS (discount): reads [(none)] -> writes [USD-OIS]
step[1] CDX-NA-IG-46 (hazard): reads [USD-OIS] -> writes [CDX-NA-IG-46]
step[2] CDX-NA-IG-46-BASE-CORR-STEP (base_correlation): reads [USD-OIS] -> writes [CDX-NA-IG-46_CORR]


## Takeaways

- Calibration envelopes keep raw market quotes outside notebook code while preserving a reproducible market-building workflow.
- `calibrate` returns a `CalibrationResult`; inspect residuals and `result.market` before downstream pricing.
- Use `dry_run` and `dependency_graph_json` for fast diagnostics before invoking slower calibration steps.

**Next:** Use the calibrated `MarketContext` in `02_pricing/pricing_fundamentals.ipynb`.


## Typed calibration result

The object returned by `calibrate` is a `CalibrationResult`: a typed wrapper with residual diagnostics, step reports, and the calibrated `MarketContext`.

In [13]:
from finstack_quant.valuations import CalibrationResult

print("CalibrationResult type:", CalibrationResult.__name__)
print("latest result is CalibrationResult:", isinstance(result, CalibrationResult))
print("available step ids:", result.step_ids)

CalibrationResult type: CalibrationResult
latest result is CalibrationResult: True
available step ids: []
